# NL to SQL

## Objetivo

Traducir consultas estructuradas en lenguaje natural a sentencias de bases de datos relacionales (SQL), integrando la librería PLY para el análisis léxico/sintáctico, y utilizando un Agente basado en Ollama (LLM) para resolver la generación del código final en la fase semántica

## Alcance

- Generación de sentencias `SELECT` especificando múltiples columnas (separadas por coma) o el comodín `TODO`.
- Implementación de filtrado mediante la palabra `CON` (cláusula `WHERE`).
- Encadenamiento de múltiples condiciones lógicas utilizando la palabra `Y` (cláusula `AND`).
- Tipos de datos soportados en filtros: Cadenas de texto (entre comillas) y Valores Numéricos.
- Operadores relacionales soportados: `IGUAL`, `MAYOR`, `MENOR`.
- Arquitectura robusta: Cuenta con un mecanismo de *fallback* determinista que genera el SQL manualmente si el servidor de Ollama se encuentra offline.

## Aspectos léxicos

| Token | Patrón / Regex | Ejemplos |
|---|---|---|
| `BUSCAR`, `TODO`, `EN`, `CON`, `Y` | Palabras reservadas exactas | BUSCAR, EN |
| `IGUAL`, `MAYOR`, `MENOR` | Palabras reservadas (Operadores) | MAYOR |
| `COMA` | `,` | , |
| `TABLA` | `[a-zA-Z_][a-zA-Z0-9_]*` | usuarios, edad |
| `CADENA` | `\"[^\"]*\"` | "activo" |
| `NUMERO` | `\d+` | 18 |

## Aspectos sintácticos

```bnf
<Programa> ::= <Consulta> | <Consulta> <Programa>
<Consulta> ::= BUSCAR <Columnas> EN TABLA 
<Consulta> ::= BUSCAR <Columnas> EN TABLA CON <Filtros> 
<Columnas> ::= TODO | TABLA | TABLA COMA <Columnas>
<Filtros> ::= <FiltroSimple> | <FiltroSimple> Y <Filtros>
<FiltroSimple> ::= TABLA <Operador> <Valor>
<Operador> ::= IGUAL | MAYOR | MENOR
<Valor> ::= CADENA | NUMERO
```

## Aspectos semánticos

La validación estructural se realiza con PLY de manera ascendente (Bottom-Up), pero **la resolución semántica final del string SQL se delega al Agente Ollama** en las reglas principales:
- `<Consulta> ::= BUSCAR <Columnas> EN TABLA`: El parser empaqueta las `columnas` y la `tabla` (TABLA) validadas y se las envía al LLM para que redacte el `SELECT`.
- `<Consulta> ::= BUSCAR <Columnas> EN TABLA CON <Filtros>`: Se empaquetan `columnas`, `tabla` y los `filtros` acumulados, y Ollama se encarga de inyectar el `WHERE` y los operadores `AND` correspondientes.
- En caso de que el LLM no responda a tiempo, la fase semántica resuelve la instrucción concatenando las variables en memoria de forma tradicional (Mecanismo de Fallback).

## Ejemplos

**Ejemplo Simple:**
- *Programa de entrada:* `BUSCAR TODO EN clientes`
- *Acciones semánticas y Traducción:* Se detecta el comodín TODO y se invoca al agente semántico, obteniendo `SELECT * FROM clientes;`

**Ejemplo Complejo:**
- *Programa de entrada:* `BUSCAR nombre, apellido EN usuarios CON edad MAYOR 18 Y estado IGUAL "activo"`
- *Acciones semánticas y Traducción:* Se procesa la recursividad de las comas en las columnas y de la 'Y' en los filtros, transformando operadores naturales a matemáticos y generando: `SELECT nombre, apellido FROM usuarios WHERE edad > 18 AND estado = 'activo';`

## Implementación

In [2]:
import requests

class AgenteOllama:
    def __init__(self, url="http://localhost:11434/api/generate", modelo="llama3"):
        self.url = url
        self.modelo = modelo

    def generar_sql(self, columnas, tabla, filtros=None):
        """
        Delega la generación del código SQL a Ollama.
        Si falla el Request (ej. servidor apagado), usa un fallback tradicional.
        """
        prompt = f"Actúa como un traductor a SQL estricto. Tabla: {tabla}. Columnas seleccionadas: {columnas}. Filtros WHERE: {filtros}. Devuelve ÚNICAMENTE la consulta SQL en texto plano, sin explicaciones y terminada en punto y coma."
        
        try:
            # Petición a la API local de Ollama
            response = requests.post(self.url, json={
                "model": self.modelo,
                "prompt": prompt,
                "stream": False
            }, timeout=3)
            if response.status_code == 200:
                return response.json()['response'].strip()
        except Exception as e:
            pass # Capturamos el error silenciosamente para activar el Fallback

        # --- FALLBACK DETERMINISTA ---
        # Esto asegura que el compilador siga funcionando aunque Ollama no esté corriendo
        if filtros:
            return f"SELECT {columnas} FROM {tabla} WHERE {filtros};"
        return f"SELECT {columnas} FROM {tabla};"

In [ ]:
import ply.lex as lex
import ply.yacc as yacc
import sys

# Evitar errores de atributo __file__ en el entorno de Jupyter Notebook
if not hasattr(sys.modules['__main__'], '__file__'):
    sys.modules['__main__'].__file__ = 'tp_traductor.py'

class TraductorNLSQL:
    # 1. ANALIZADOR LÉXICO
    tokens = (
        'BUSCAR', 'TODO', 'EN', 'CON', 'Y', 'IGUAL', 'MAYOR', 'MENOR', 
        'COMA', 'TABLA', 'CADENA', 'NUMERO'
    )

    t_COMA = r','

    def t_CADENA(self, t):
        r'\"[^\"]*\"'
        t.value = t.value.replace('"', "'") # Ajustamos comillas para SQL
        return t

    def t_NUMERO(self, t):
        r'\d+'
        return t

    def t_TABLA(self, t):
        r'[a-zA-Z_][a-zA-Z0-9_]*'
        reservadas = {
            'BUSCAR': 'BUSCAR', 'TODO': 'TODO', 'EN': 'EN', 'CON': 'CON', 
            'Y': 'Y', 'IGUAL': 'IGUAL', 'MAYOR': 'MAYOR', 'MENOR': 'MENOR'
        }
        t.type = reservadas.get(t.value.upper(), 'TABLA')
        return t

    t_ignore = ' \t'

    def t_newline(self, t):
        r'\n+'
        t.lexer.lineno += len(t.value)

    def t_error(self, t):
        print(f"Error léxico: palabra '{t.value[0]}' no reconocida en línea {t.lexer.lineno}")
        t.lexer.skip(1)

    # 2. ANALIZADOR SINTÁCTICO Y SEMÁNTICO
    def p_programa_multiple(self, p):
        '''programa : consulta programa'''
        p[0] = p[1] + "\n" + p[2]

    def p_programa_simple(self, p):
        '''programa : consulta'''
        p[0] = p[1]

    # -- INTEGRACIÓN DE OLLAMA EN LAS ACCIONES SEMÁNTICAS PRINCIPALES --
    def p_consulta_simple(self, p):
        '''consulta : BUSCAR columnas EN TABLA'''
        # Enviamos los tokens procesados al Agente para su traducción final
        p[0] = self.agente.generar_sql(columnas=p[2], tabla=p[4].lower())

    def p_consulta_filtro(self, p):
        '''consulta : BUSCAR columnas EN TABLA CON filtros'''
        p[0] = self.agente.generar_sql(columnas=p[2], tabla=p[4].lower(), filtros=p[6])

    # -- REGLAS RECURSIVAS --
    def p_columnas_todo(self, p):
        '''columnas : TODO'''
        p[0] = "*"

    def p_columnas_lista_multiple(self, p):
        '''columnas : TABLA COMA columnas'''
        p[0] = f"{p[1].lower()}, {p[3]}"

    def p_columnas_lista_simple(self, p):
        '''columnas : TABLA'''
        p[0] = p[1].lower()

    def p_filtros_multiple(self, p):
        '''filtros : filtro_simple Y filtros'''
        p[0] = f"{p[1]} AND {p[3]}"

    def p_filtros_simple(self, p):
        '''filtros : filtro_simple'''
        p[0] = p[1]

    def p_filtro_simple(self, p):
        '''filtro_simple : TABLA operador valor'''
        p[0] = f"{p[1].lower()} {p[2]} {p[3]}"

    def p_operador_igual(self, p):
        '''operador : IGUAL'''
        p[0] = "="

    def p_operador_mayor(self, p):
        '''operador : MAYOR'''
        p[0] = ">"

    def p_operador_menor(self, p):
        '''operador : MENOR'''
        p[0] = "<"

    def p_valor_cadena(self, p):
        '''valor : CADENA'''
        p[0] = p[1]

    def p_valor_numero(self, p):
        '''valor : NUMERO'''
        p[0] = p[1]

    def p_error(self, p):
        if p:
            print(f"Error de sintaxis cerca de '{p.value}'")
        else:
            print("Error de sintaxis al final del código")

    # 3. CONSTRUCTOR
    def __init__(self):
        self.agente = AgenteOllama() # Instanciamos el LLM
        self.lexer = lex.lex(object=self)
        self.parser = yacc.yacc(module=self, write_tables=False)

    def traducir(self, codigo):
        return self.parser.parse(codigo, lexer=self.lexer)

## Tests

In [5]:
compilador = TraductorNLSQL()

print("========================================")
print("       TESTS: TRADUCTOR NL-SQL          ")
print("========================================")

print("\n--- 1. EJEMPLO SIMPLE ---")
codigo_simple = "BUSCAR TODO EN clientes"
print("Entrada:", codigo_simple)
print("Salida:", compilador.traducir(codigo_simple))

print("\n--- 2. EJEMPLO COMPLEJO (Múltiples columnas y lógica AND) ---")
codigo_complejo = "BUSCAR nombre, apellido EN usuarios CON edad MAYOR 18 Y estado IGUAL \"activo\""
print("Entrada:", codigo_complejo)
print("Salida:", compilador.traducir(codigo_complejo))

print("\n--- 3. TEST DE ERROR SINTÁCTICO (Falta coma) ---")
codigo_error = "BUSCAR nombre apellido EN clientes"
print("Entrada:", codigo_error)
compilador.traducir(codigo_error)

print("\n--- 4. TEST MULTILÍNEA ---")
codigo_multi = '''BUSCAR TODO EN productos CON stock MENOR 10
BUSCAR id, total EN ventas CON monto MAYOR 50000'''
print("Entrada:\n" + codigo_multi)
print("\nSalida:\n" + compilador.traducir(codigo_multi))

Generating LALR tables


       TESTS: TRADUCTOR NL-SQL          

--- 1. EJEMPLO SIMPLE ---
Entrada: BUSCAR TODO EN clientes
Salida: SELECT * FROM clientes;

--- 2. EJEMPLO COMPLEJO (Múltiples columnas y lógica AND) ---
Entrada: BUSCAR nombre, apellido EN usuarios CON edad MAYOR 18 Y estado IGUAL "activo"
Salida: SELECT nombre, apellido FROM usuarios WHERE edad > 18 AND estado = 'activo';

--- 3. TEST DE ERROR SINTÁCTICO (Falta coma) ---
Entrada: BUSCAR nombre apellido EN clientes
Error de sintaxis cerca de 'apellido'

--- 4. TEST MULTILÍNEA ---
Entrada:
BUSCAR TODO EN productos CON stock MENOR 10
BUSCAR id, total EN ventas CON monto MAYOR 50000

Salida:
SELECT * FROM productos WHERE stock < 10;
SELECT id, total FROM ventas WHERE monto > 50000;


## Conclusiones

Realicé un traductor usando un metacompilador en Python que traduce oraciones en español a SQL usando PLY para leer y validar el texto y un agente con Ollama para generar el código final, con un sistema de respaldo por si se cuelga la IA.

Me costo mucho armar la gramática para que entienda varias columnas separadas por comas y encadenar condiciones con la palabra "Y", lidiando con los cambios sobre la marcha.

Me sirvio para entender como funciona la base de un lenguaje de programación: pasar de texto humano a código ejecutable real.